# 2.1 · Training a mean-difference probe for your submission

This is the advanced version of [`2-predicting.ipynb`](2-predicting.ipynb). There, we trained a logistic-regression probe to *understand* how white-box deception detection works. Here, we train the probe you can actually **submit** to the competition.

The detector is a **mean-difference probe** (the "mean" / mass-mean-shift probe from Apollo Research's *Detecting Strategic Deception Using Linear Probes*, [arXiv:2502.03407](https://arxiv.org/abs/2502.03407)). Given residual-stream activations pooled over the assistant's response, you take the **difference of the class means**:

$$ \mathbf{d} \;=\; \underbrace{\overline{\mathbf{h}}_{\text{deceptive}}}_{\text{mean over lying responses}} \;-\; \underbrace{\overline{\mathbf{h}}_{\text{honest}}}_{\text{mean over honest responses}} $$

and score a new response by projecting its activation onto that direction: $s(\mathbf{h}) = \mathbf{h}\cdot\mathbf{d}$. No gradient descent, no regularization — two means and a subtraction. In the paper this probe is competitive with (and often more robust than) logistic regression, and its direction is interpretable as "the honesty axis." It's a faithful port of the `MMSDetector` from the [deception-detection](https://github.com/Cadenza-Labs/deception-detection) repo, whose entire `fit` is `direction = positive_acts.mean(0) - negative_acts.mean(0)`.

**The data, tokenization, and activation extraction are the same as notebook 2** — only the *probe* and the *export* change, which keeps the two directly comparable. Our plan:

1. Define the mean-difference probe and sanity-check it on synthetic data.
2. Pull labeled honest/deceptive conversations from Liars' Bench, balanced across scenarios, **holding one scenario out** for transfer.
3. Tokenize with the model's chat template and extract pooled activations at every layer (one remote pass).
4. Fit the probe per layer; pick the layer on a held-out **validation** split.
5. Report honest **cross-scenario transfer** on a held-out **test** split, and compare to logistic regression.
6. **Export the trained probe** to `mean_probe.npz` for the competition submission.

> This notebook traces a model remotely on NDIF. If you haven't set up access, do [`1-setup.ipynb`](1-setup.ipynb) first.

## 0️⃣ Setup

Install the hackathon build of `nnsight` (PEFT/LoRA + remote NDIF), then point `nnsight` at the hackathon host and load your credentials from the environment — exactly as in notebooks 1 and 2.

In [ ]:
# nnsight — hackathon build (PEFT/LoRA + remote NDIF support)
!pip install -q git+https://github.com/ndif-team/nnsight.git@hackathon/peft
!pip install -q peft
!pip install -q --upgrade torchao transformers

from IPython.display import clear_output
clear_output()
print("Setup complete.")

In [ ]:
import os
from nnsight import CONFIG

NDIF_API_KEY = os.environ.get("NDIF_API_KEY", "YOUR_NDIF_API_KEY")
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN")  # gated dataset/model access

CONFIG.set_default_api_key(NDIF_API_KEY)

# Competition models + LoRA imposters are served on the hackathon NDIF instance.
CONFIG.API.HOST = "http://ndif-hackathon.duckdns.org:8001"

repo_id = "google/gemma-3-27b-it"   # the model whose activations we probe (and submit against)

**Why Gemma-3-27B?** The competition submission traces this exact model, so the probe we export must be trained on *its* activations. Because Gemma-3 is multimodal, we load it with `VisionLanguageModel` (not `LanguageModel`) — its text decoder is nested under `.language_model`, which matters when we reach for hidden states.

The Liars' Bench **train** data is gated, so authenticate with HuggingFace if your token is set. The `"Hello world"` trace is a cheap smoke test: if the remote deployment is cold or down, this fails fast before you spend compute on the rest of the pipeline.

In [ ]:
from huggingface_hub import login

# Authenticate for the gated Liars' Bench dataset (no-op if you haven't set HF_TOKEN).
if os.environ["HF_TOKEN"] not in ("", "YOUR_HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])

In [ ]:
from nnsight import VisionLanguageModel
from datasets import load_dataset, get_dataset_config_names

model = VisionLanguageModel(repo_id)

with model.trace("Hello world", remote=True):
    _ = model.output.save()
print("Remote trace OK — the model is reachable.")

## 1️⃣ The probe

Here's the entire method. `fit` computes the difference of class means; `decision_function` projects onto that direction. Two conveniences don't change the paper's method:

- **`normalize`** — per-dimension z-score before the mean difference. This only rescales the score, so **ranking metrics like AUROC are unchanged**; it mainly makes the `bias` threshold better behaved for hard accuracy.
- **`bias`** — the midpoint between the honest and deceptive mean projections, a sensible default decision threshold without a separate calibration step.

> **Convention:** positive class (`y == 1`) = **deceptive**, negative (`y == 0`) = **honest**, so a *higher* score means *more deceptive* — matching `MMSDetector`, where `positive_acts` are the deceptive activations.

In [ ]:
import numpy as np

class MeanDifferenceProbe:
    """Difference-of-means deception probe (the "mean" / MMS probe).

    Faithful port of MMSDetector from the deception-detection repo:
        direction = mean(deceptive activations) - mean(honest activations)
        score(h)  = h . direction
    Operates on one pooled activation vector per example.
    """
    def __init__(self, normalize: bool = True):
        self.normalize = normalize
        self.direction = self.mu = self.sigma = self.bias = None

    def _norm(self, X):
        return (X - self.mu) / self.sigma if self.normalize else X

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        if self.normalize:
            self.mu = X.mean(axis=0)
            self.sigma = X.std(axis=0) + 1e-8
        Xn = self._norm(X)
        pos_mean = Xn[y == 1].mean(axis=0)   # deceptive
        neg_mean = Xn[y == 0].mean(axis=0)   # honest
        self.direction = pos_mean - neg_mean
        # Natural threshold: midpoint of the two class-mean projections.
        s = Xn @ self.direction
        self.bias = 0.5 * (s[y == 1].mean() + s[y == 0].mean())
        return self

    def decision_function(self, X):
        return self._norm(np.asarray(X, dtype=np.float64)) @ self.direction - self.bias

    def predict(self, X):
        return (self.decision_function(X) > 0).astype(int)

    def predict_proba(self, X):
        # logistic squashing of the score, for a probability-like column (col 1 = deceptive)
        p1 = 1.0 / (1.0 + np.exp(-self.decision_function(X)))
        return np.stack([1 - p1, p1], axis=1)

### Sanity-check on synthetic data

Before touching the model, confirm the math on data with a *planted* deceptive direction. The probe should (a) score the held-out split well above chance, (b) recover the planted direction (high cosine similarity), and (c) give an AUROC invariant to `normalize` (a monotone rescaling). This cell needs no GPU and no NDIF — it verifies the method itself.

In [ ]:
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

_rng = np.random.default_rng(0)
_D, _n = 64, 200
_d_true = _rng.normal(size=_D); _d_true /= np.linalg.norm(_d_true)
_Xh = _rng.normal(size=(_n, _D))                       # honest
_Xd = _rng.normal(size=(_n, _D)) + 1.5 * _d_true       # deceptive: shifted along planted axis
_X = np.vstack([_Xh, _Xd]); _y = np.r_[np.zeros(_n), np.ones(_n)]
_p = _rng.permutation(len(_y)); _X, _y = _X[_p], _y[_p]
_ntr = int(0.7 * len(_y))

_probe = MeanDifferenceProbe(normalize=True).fit(_X[:_ntr], _y[:_ntr])
_s = _probe.decision_function(_X[_ntr:])
_cos = abs(np.dot(_probe.direction / np.linalg.norm(_probe.direction), _d_true))
_s_nonorm = MeanDifferenceProbe(normalize=False).fit(_X[:_ntr], _y[:_ntr]).decision_function(_X[_ntr:])
print(f"AUROC (norm):    {roc_auc_score(_y[_ntr:], _s):.3f}")
print(f"AUROC (no norm): {roc_auc_score(_y[_ntr:], _s_nonorm):.3f}   <- same ranking, AUROC invariant")
print(f"bal-acc:         {balanced_accuracy_score(_y[_ntr:], _probe.predict(_X[_ntr:])):.3f}")
print(f"cos(learned, planted direction): {_cos:.3f}")

🧠 **Takeaway:** the probe is two means and a subtraction, and on clean data it recovers the planted direction exactly. Everything that follows is about *feeding it the right activations* — which is where the model, the data, and the tokenization come in.

## 2️⃣ The data

We use the same Liars' Bench data as notebook 2 — see [`2-predicting.ipynb`](2-predicting.ipynb) §1 for the full scenario taxonomy and what counts as a "lie." Here we move faster and focus on the two things that matter for a *submittable* probe:

- **Balance** across scenarios and classes, so the class means aren't dominated by one setup.
- **A held-out scenario**, so our transfer number is honest.

First, list the configs.

In [ ]:
configs = get_dataset_config_names("Cadenza-Labs/liars-bench")
print("Available configs:", configs)

A quick look at one config grounds the schema — fields, generating model, and the `deceptive` label.

In [ ]:
ds = load_dataset("Cadenza-Labs/liars-bench", data_dir="instructed-deception")
print(ds)

split_name = list(ds.keys())[0]
ex = ds[split_name][0]
print("Keys:", list(ex.keys()))
print("Generator model:", ex["model"])
print("Deceptive label:", ex["deceptive"])
print("\nConversation:")
for msg in ex["messages"]:
    snippet = msg["content"][:300].replace("\n", " ")
    print(f"  [{msg['role']}]: {snippet}")

### Combine several configs

A probe trained on one scenario latches onto *that scenario's* surface features rather than deception itself. We pull several configs and tag each row with its origin, projecting to a uniform column set before concatenating. One config (`insider-trading`) is included here but **reserved as the held-out transfer target** below.

In [ ]:
from datasets import concatenate_datasets

CONFIGS = [
    "soft-trigger",
    "harm-pressure-choice",
    "instructed-deception",
    "insider-trading",   # reserved below as the held-out transfer config
]
NEEDED_COLS = ["model", "messages", "deceptive"]

per_config = []
for cfg in CONFIGS:
    try:
        d = load_dataset("Cadenza-Labs/liars-bench", data_dir=cfg)
    except Exception as e:
        print(f"  [skip] {cfg}: {type(e).__name__}: {str(e)[:120]}")
        continue
    sub = d[next(iter(d.keys()))]
    cols_present = [c for c in NEEDED_COLS if c in sub.column_names]
    if set(NEEDED_COLS) - set(cols_present):
        print(f"  [skip] {cfg}: missing {set(NEEDED_COLS) - set(cols_present)}")
        continue
    sub = sub.select_columns(cols_present).add_column("config", [cfg] * len(sub))
    per_config.append(sub)
    print(f"  loaded {cfg}: {len(sub)} rows")

combined = concatenate_datasets(per_config)
ds = {"train": combined}
print()
print(combined)

### Coverage by generator

A probe reads the activations of a *specific* model, so its training labels must come from that same model's generations. We filter to `GENERATOR` (Gemma-3-27B, matching the model we probe) and check which configs it covers with both classes.

In [ ]:
import pandas as pd

GENERATOR = "gemma-3-27b-it"   # must match the model whose activations we probe

df = ds["train"].to_pandas()
gen_df = df[df["model"] == GENERATOR]
print(f"Rows generated by {GENERATOR}: {len(gen_df)}\n")

cov = gen_df.groupby("config").agg(
    total=("deceptive", "count"),
    deceptive=("deceptive", "sum"),
    honest=("deceptive", lambda s: (~s).sum()),
    deceptive_rate=("deceptive", "mean"),
).round(3)
print(cov.to_string())

### Balanced train set + a held-out scenario

We sample a balanced number of deceptive/honest examples per config (so no scenario or class dominates the mean), and **reserve one whole config for transfer evaluation**. Holding out an entire *scenario* — not just some rows — is the honest test of whether the probe learns "deception" rather than one setup's surface features.

The selection is adaptive: only configs the generator actually covers (both classes) are used. If only one such config exists, the held-out set falls back to a disjoint *split* of that config, flagged as in-distribution generalization rather than cross-scenario transfer.

In [ ]:
HOLDOUT_CONFIG = "insider-trading"   # cross-scenario transfer target (if the generator covers it)
N_PER_CONFIG_PER_LABEL = 100         # balanced cap per (config, label)
SEED = 42
rng = np.random.default_rng(SEED)

gen_df = gen_df.reset_index(drop=True)
gen_df["orig_idx"] = gen_df.index

def covered(frame):
    """configs with >=1 deceptive AND >=1 honest example"""
    out = []
    for cfg in frame["config"].unique():
        s = frame[frame["config"] == cfg]["deceptive"]
        if s.sum() > 0 and (~s).sum() > 0:
            out.append(cfg)
    return out

train_pool = gen_df[gen_df["config"] != HOLDOUT_CONFIG]
train_configs = covered(train_pool)
holdout_pool = gen_df[gen_df["config"] == HOLDOUT_CONFIG]
cross_scenario = HOLDOUT_CONFIG in covered(gen_df) and len(train_configs) >= 1

if not cross_scenario:
    # Fallback: not enough generator coverage to hold out a whole scenario.
    # Use a disjoint split of the covered configs instead.
    all_cov = covered(gen_df)
    assert all_cov, f"No config has both classes for generator {GENERATOR}."
    print("[note] Not enough coverage for cross-scenario holdout; "
          "using a disjoint in-distribution split instead.")
    train_pool = gen_df[gen_df["config"].isin(all_cov)]
    train_configs = all_cov

def sample_balanced(pool, configs, n_cap):
    parts = []
    for cfg in configs:
        sub = pool[pool["config"] == cfg]
        dec = sub[sub["deceptive"]]["orig_idx"].values
        hon = sub[~sub["deceptive"]]["orig_idx"].values
        nd, nh = min(n_cap, len(dec)), min(n_cap, len(hon))
        parts.append(rng.choice(dec, nd, replace=False))
        parts.append(rng.choice(hon, nh, replace=False))
        print(f"  {cfg}: {nd} deceptive + {nh} honest")
    idx = np.concatenate(parts); rng.shuffle(idx)
    return idx

print(f"Training configs: {train_configs}")
train_idx = sample_balanced(train_pool, train_configs, N_PER_CONFIG_PER_LABEL)

if cross_scenario:
    print(f"\nHeld-out (cross-scenario) config: {HOLDOUT_CONFIG}")
    hd = holdout_pool[holdout_pool["deceptive"]]["orig_idx"].values
    hh = holdout_pool[~holdout_pool["deceptive"]]["orig_idx"].values
    n = min(len(hd), len(hh), 150)
    holdout_idx = np.concatenate([rng.choice(hd, n, replace=False),
                                  rng.choice(hh, n, replace=False)])
    rng.shuffle(holdout_idx)
    # Ensure train and holdout never overlap (different configs guarantees this).
    holdout_idx = np.array([i for i in holdout_idx if i not in set(train_idx)])
else:
    # Disjoint in-distribution split.
    from sklearn.model_selection import train_test_split as _tts
    train_idx, holdout_idx = _tts(
        train_idx,
        test_size=0.4,
        random_state=SEED,
        stratify=gen_df.loc[train_idx, "deceptive"].values,
    )

train_examples = [gen_df.iloc[int(i)].to_dict() for i in train_idx]
train_labels   = np.array([int(e["deceptive"]) for e in train_examples])
holdout_examples = [gen_df.iloc[int(i)].to_dict() for i in holdout_idx]
holdout_labels   = np.array([int(e["deceptive"]) for e in holdout_examples])

print(f"\nTrain:    {len(train_examples)} ({train_labels.sum()} deceptive / "
      f"{len(train_labels)-train_labels.sum()} honest)")
print(f"Held-out: {len(holdout_examples)} ({holdout_labels.sum()} deceptive / "
      f"{len(holdout_labels)-holdout_labels.sum()} honest)")

## 3️⃣ Extract activations

This is the same recipe as notebook 2 §1.3: wrap each conversation in the model's chat template, locate the assistant **response span**, run the tokens through the model on NDIF, and **mean-pool the residual stream over the response tokens only**. Getting this exactly right matters twice over — the submission notebook must reproduce it byte-for-byte, or the probe sees out-of-distribution activations.

### Tokenize with the chat template

Gemma's template doesn't accept a standalone `system` turn, so any leading system message is merged into the first user message. The `tokenize_raw_response_range` alternative skips the template (raw concat) — handy as an A/B to test whether the template actually helps.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "google/gemma-3-27b-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
EOT_ID = tokenizer.convert_tokens_to_ids("<end_of_turn>")   # Gemma chat turn terminator

USE_CHAT_TEMPLATE = True   # flip to False to use raw concat tokens instead

def tokenize_with_response_range(messages):
    """Chat-template path: (input_ids, response_start, response_end); end excludes trailing eot."""
    norm, pending_system = [], None
    for m in messages:
        if m["role"] == "system" and not norm:
            pending_system = m["content"]; continue
        if not norm and m["role"] == "user" and pending_system:
            norm.append({"role": "user", "content": pending_system + "\n\n" + m["content"]})
            pending_system = None; continue
        norm.append(m)
    full_ids = tokenizer.apply_chat_template(
        norm, tokenize=True, add_generation_prompt=False, return_dict=True)["input_ids"]
    prefix_ids = tokenizer.apply_chat_template(
        norm[:-1], tokenize=True, add_generation_prompt=True, return_dict=True)["input_ids"]
    start, end = len(prefix_ids), len(full_ids)
    while end > start and full_ids[end - 1] == EOT_ID:   # strip trailing end-of-turn
        end -= 1
    return full_ids, start, end

def tokenize_raw_response_range(messages):
    """Raw concat: BOS + non-assistant texts + assistant text. No role headers, no eot."""
    *prefix_msgs, last = messages
    assert last["role"] == "assistant", "expected last message to be assistant"
    prompt_text = "\n".join(m["content"] for m in prefix_msgs)
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=True)
    response_ids = tokenizer.encode(last["content"], add_special_tokens=False)
    return prompt_ids + response_ids, len(prompt_ids), len(prompt_ids) + len(response_ids)

tokenize_fn = tokenize_with_response_range if USE_CHAT_TEMPLATE else tokenize_raw_response_range
print(f"Tokenize mode: {'chat-template' if USE_CHAT_TEMPLATE else 'raw'}\n")

train_tok = [tokenize_fn(e["messages"]) for e in train_examples]
lens = [len(t[0]) for t in train_tok]
rlens = [t[2] - t[1] for t in train_tok]
print(f"Token length:    min={min(lens)} max={max(lens)} mean={np.mean(lens):.0f}")
print(f"Response length: min={min(rlens)} max={max(rlens)} mean={np.mean(rlens):.0f}")

### Pool every layer in one remote pass

We don't yet know which layer carries the cleanest deception signal, so we pool **every** layer's residual stream in a single remote trace — mean-pooling over the response span *inside* the trace, so only a `(batch, hidden)` vector per layer is shipped back instead of the full `(batch, seq, hidden)` residual stream (tens of GB across all layers). It's the **same forward pass**: sweeping every layer costs essentially the same remote compute as extracting one.

> **Architecture note:** Gemma-3's text decoder is nested under `.language_model`, so the residual stream lives at `model.model.language_model.layers[L].output[0]` (the `[0]` selects the hidden state from the layer's output tuple). Text-only models use `model.model.layers[L].output[0]`.

In [ ]:
import torch
from IPython.display import clear_output

text_cfg = getattr(model.config, "text_config", model.config)   # multimodal config nests text_config
N_LAYERS = text_cfg.num_hidden_layers
LAYER_STRIDE = 1
LAYERS = list(range(0, N_LAYERS, LAYER_STRIDE))
# LAYERS = [N_LAYERS // 2]    # <- uncomment for a single-layer smoke test first
BATCH_SIZE = 2
MAX_SEQ_LEN = 300
PAD_ID = EOT_ID              # attention_mask=0 at pad positions, so any valid id works
print(f"Sweeping {len(LAYERS)} layers (0..{N_LAYERS-1}, stride {LAYER_STRIDE})")

def collate(batch):
    max_len = max(len(t[0]) for t in batch)
    B = len(batch)
    input_ids = torch.full((B, max_len), PAD_ID, dtype=torch.long)
    attn_mask = torch.zeros((B, max_len), dtype=torch.long)
    resp_mask = torch.zeros((B, max_len), dtype=torch.float32)
    for i, (ids, s, e) in enumerate(batch):
        input_ids[i, :len(ids)] = torch.tensor(ids)
        attn_mask[i, :len(ids)] = 1
        resp_mask[i, s:e] = 1.0
    return input_ids, attn_mask, resp_mask

def extract_all_layers(tok_list, layers, batch_size=BATCH_SIZE):
    """Run tok_list through the remote model once per batch, mean-pooling the response span at
    every layer inside the trace. Returns {layer: (N, hidden) ndarray}."""
    out = {L: [] for L in layers}
    n_batches = (len(tok_list) + batch_size - 1) // batch_size
    for b in range(n_batches):
        batch = tok_list[b*batch_size:(b+1)*batch_size]
        input_ids, attn_mask, resp_mask = collate(batch)
        with model.trace({"input_ids": input_ids, "attention_mask": attn_mask}, remote=True):
            pooled = []
            for L in layers:
                h = model.model.language_model.layers[L].output[0]   # (B, T, D)
                # resp_mask is built on CPU locally; h lives on GPU (and may differ per layer)
                mask = resp_mask.to(h.device).unsqueeze(-1)
                denom = resp_mask.to(h.device).sum(dim=1, keepdim=True).clamp(min=1)
                layer_batch = (h * mask).sum(dim=1) / denom
                pooled.append(layer_batch.detach().cpu())
            # nnsight only injects direct assignments (x = ...save()), not saved[L] = ...
            all_layers = torch.stack(pooled, dim=0).save()   # (num_layers, B, hidden), on CPU
        batch_feats = all_layers.float().numpy()
        for i, L in enumerate(layers):
            out[L].append(batch_feats[i])
        clear_output()
        print(f"  batch {b+1}/{n_batches} (max_len={input_ids.shape[1]})")
    return {L: np.concatenate(out[L], axis=0) for L in layers}

# Drop outlier-long sequences, then extract.
keep = [(t, l) for t, l in zip(train_tok, train_labels) if len(t[0]) <= MAX_SEQ_LEN]
train_tok = [t for t, _ in keep]
y = np.array([l for _, l in keep])
print(f"Kept {len(train_tok)}/{len(train_labels)} train examples (<= {MAX_SEQ_LEN} tokens)")

X_by_layer = extract_all_layers(train_tok, LAYERS)
print(f"Extracted {len(LAYERS)} layers | X[{LAYERS[0]}]: {X_by_layer[LAYERS[0]].shape} | y: {y.shape}")

## 4️⃣ Train the probe on every layer

Fit one `MeanDifferenceProbe` per layer and record each layer's **in-distribution** holdout AUROC (80/20 stratified). Each probe is then refit on all training rows and stored.

**Don't select the layer from this table.** In-distribution AUROC is near-saturated and doesn't predict transfer — the held-out scenario decides that, in Section 5.

In [ ]:
from sklearn.model_selection import train_test_split

probes_by_layer = {}
indist_rows = []
for L in LAYERS:
    XL = X_by_layer[L]
    X_tr, X_te, y_tr, y_te = train_test_split(XL, y, test_size=0.2, stratify=y, random_state=42)
    clf = MeanDifferenceProbe(normalize=True).fit(X_tr, y_tr)
    indist_rows.append({
        "layer": L,
        "indist_auroc": roc_auc_score(y_te, clf.decision_function(X_te)),
        "indist_bacc":  balanced_accuracy_score(y_te, clf.predict(X_te)),
    })
    probes_by_layer[L] = MeanDifferenceProbe(normalize=True).fit(XL, y)   # refit on all rows

indist_df = pd.DataFrame(indist_rows).set_index("layer")
print(indist_df.round(3).to_string())
print(f"\nBest in-dist layer: {indist_df['indist_auroc'].idxmax()} "
      f"(AUROC {indist_df['indist_auroc'].max():.3f}) -- informational only; transfer decides below.")

### Visualize the honesty axis

At a representative mid-late layer, plot the distribution of probe scores for honest vs. deceptive responses. Clean separation here means the difference-of-means direction genuinely captures a deception signal in the residual stream.

In [ ]:
import matplotlib.pyplot as plt

L_show = LAYERS[len(LAYERS) * 2 // 3]   # a mid-late layer
probe_show = probes_by_layer[L_show]
scores = probe_show.decision_function(X_by_layer[L_show])

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores[y == 0], bins=30, alpha=0.6, label="honest", color="tab:blue")
ax.hist(scores[y == 1], bins=30, alpha=0.6, label="deceptive", color="tab:red")
ax.axvline(0, color="k", ls="--", lw=1, label="threshold")
ax.set_xlabel(f"mean-probe score at layer {L_show}  (h . direction - bias)")
ax.set_ylabel("count"); ax.set_title("Honesty axis: honest vs. deceptive responses")
ax.legend(); plt.tight_layout(); plt.show()

## 5️⃣ Cross-scenario transfer: the real test

Does the probe flag deception in a config it **never trained on**? That transfer number — not in-distribution accuracy — is what the competition rewards. (See [`2-predicting.ipynb`](2-predicting.ipynb) §3 for what AUROC means and why below-0.5 signals an *inverted* probe.)

We extract activations for the held-out examples (same forward-pass extraction), then split them into a **val** half (to pick the layer) and a **test** half (the reported number), so the layer isn't chosen on the same rows it's evaluated on. Reading the per-layer curve:

- **Signal is real:** the best layers cluster together and val ≈ test.
- **It's noise:** the argmax jumps around and val/test disagree — then no single layer truly transfers, and the honest takeaway is that the probe doesn't generalize to this scenario.

In [ ]:
# Tokenize + extract the held-out set with the SAME path as training.
holdout_tok = [tokenize_fn(e["messages"]) for e in holdout_examples]
keep = [(t, l) for t, l in zip(holdout_tok, holdout_labels) if len(t[0]) <= MAX_SEQ_LEN]
holdout_tok = [t for t, _ in keep]
y_hold = np.array([l for _, l in keep])
print(f"Kept {len(holdout_tok)}/{len(holdout_labels)} held-out examples")

Xh_by_layer = extract_all_layers(holdout_tok, LAYERS)

val_idx, test_idx = train_test_split(
    np.arange(len(y_hold)), test_size=0.5, stratify=y_hold, random_state=42)
print(f"val: {len(val_idx)} (dec {y_hold[val_idx].sum()}) | "
      f"test: {len(test_idx)} (dec {y_hold[test_idx].sum()})")

In [ ]:
rows = []
for L in LAYERS:
    s = probes_by_layer[L].decision_function(Xh_by_layer[L])
    rows.append({
        "layer": L,
        "indist_auroc":  indist_df.loc[L, "indist_auroc"],
        "val_auroc":     roc_auc_score(y_hold[val_idx],  s[val_idx]),
        "test_auroc":    roc_auc_score(y_hold[test_idx], s[test_idx]),
    })
sweep = pd.DataFrame(rows).set_index("layer")

best_layer = int(sweep["val_auroc"].idxmax())
probe = probes_by_layer[best_layer]   # the probe you'd deploy
print(sweep.round(3).to_string())
print(f"\nSelected layer {best_layer} on val (AUROC {sweep.loc[best_layer, 'val_auroc']:.3f})")
print(f"  -> honest transfer = TEST AUROC {sweep.loc[best_layer, 'test_auroc']:.3f}")
oracle = int(sweep["test_auroc"].idxmax())
print(f"  (oracle best test = {sweep['test_auroc'].max():.3f} at layer {oracle})")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sweep.index, sweep["indist_auroc"], marker=".", label="in-dist holdout")
ax.plot(sweep.index, sweep["val_auroc"],    marker=".", label="held-out val (selection)")
ax.plot(sweep.index, sweep["test_auroc"],   marker=".", label="held-out test (reported)")
ax.axhline(0.5, color="gray", ls="--", lw=1, label="chance")
ax.axvline(best_layer, color="red", ls=":", lw=1, label=f"selected (layer {best_layer})")
ax.set_xlabel("layer"); ax.set_ylabel("AUROC"); ax.set_ylim(0, 1)
ax.set_title("Mean-difference probe: in-distribution vs. cross-scenario transfer")
ax.legend(loc="lower right", fontsize=8); plt.tight_layout(); plt.show()

### Mean probe vs. logistic regression

Same activations, same splits — only the probe differs. This situates the mean-difference probe against the logistic-regression probe from notebook 2. The mean probe has *no* hyperparameters and a single closed-form direction; LR fits a regularized boundary. On transfer the two are often close, and the mean probe is frequently the more robust of the two — a key observation in the paper.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

comp = []
for L in LAYERS:
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0)).fit(X_by_layer[L], y)
    s_lr = lr.decision_function(Xh_by_layer[L])
    comp.append({
        "layer": L,
        "mean_test":  sweep.loc[L, "test_auroc"],
        "lr_test":    roc_auc_score(y_hold[test_idx], s_lr[test_idx]),
    })
comp_df = pd.DataFrame(comp).set_index("layer")

# Each detector selects its own best layer on val, report that layer's test AUROC (apples-to-apples).
lr_val = {L: roc_auc_score(y_hold[val_idx],
            make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
            .fit(X_by_layer[L], y).decision_function(Xh_by_layer[L])[val_idx]) for L in LAYERS}
lr_best = max(lr_val, key=lr_val.get)
print(f"Mean probe : layer {best_layer} -> test AUROC {sweep.loc[best_layer, 'test_auroc']:.3f}")
print(f"LR probe   : layer {lr_best} -> test AUROC {comp_df.loc[lr_best, 'lr_test']:.3f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(comp_df.index, comp_df["mean_test"], marker=".", label="mean-difference probe")
ax.plot(comp_df.index, comp_df["lr_test"],   marker=".", label="logistic regression")
ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.set_xlabel("layer"); ax.set_ylabel("held-out test AUROC"); ax.set_ylim(0, 1)
ax.set_title("Mean-difference vs. logistic-regression probe (cross-scenario transfer)")
ax.legend(loc="lower right", fontsize=8); plt.tight_layout(); plt.show()

## 6️⃣ Export the probe for the competition

This is the payoff. We save the trained probe to a tiny `mean_probe.npz` (a few KB): the **direction**, the feature-normalization stats (`mu`, `sigma`), the decision midpoint (`bias`), the training-score spread (`score_std`, used to turn scores into well-spread probabilities), and the metadata needed to reproduce the *exact same* activation extraction at inference (`model_id`, `layer`, `use_chat_template`, `max_seq_len`).

Why train here and ship weights? In the [Aletheia's Quest](../Aletheias-Quest-Competition) sandbox the eval set is **label-free** and you can't retrain — so the probe is trained **here, offline**, and the submission notebook ([`submissions/mean_probe.ipynb`](../Aletheias-Quest-Competition/submissions/mean_probe.ipynb)) loads these weights, re-runs the *identical* extraction on the eval transcripts, scores them, and writes `submission.csv`. Copy the file next to that notebook:

```
cp mean_probe.npz ../Aletheias-Quest-Competition/submissions/mean_probe.npz
```

> **Keep extraction in lockstep.** The submission must extract activations exactly as we did above — same `layer`, `use_chat_template`, `max_seq_len`, response-span pooling, and the same `.output[0]` hidden-state path. Any drift puts the eval activations out of distribution relative to the probe and silently tanks your score.

In [ ]:
best_probe = probes_by_layer[best_layer]
s_train = best_probe._norm(X_by_layer[best_layer]) @ best_probe.direction   # training scores

np.savez(
    "mean_probe.npz",
    direction=best_probe.direction.astype(np.float32),
    mu=(best_probe.mu if best_probe.mu is not None
        else np.zeros_like(best_probe.direction)).astype(np.float32),
    sigma=(best_probe.sigma if best_probe.sigma is not None
           else np.ones_like(best_probe.direction)).astype(np.float32),
    normalize=np.array(bool(best_probe.normalize)),
    bias=np.array(float(best_probe.bias), dtype=np.float32),
    score_std=np.array(float(s_train.std()) + 1e-8, dtype=np.float32),
    layer=np.array(int(best_layer)),
    model_id=np.array(repo_id),
    use_chat_template=np.array(bool(USE_CHAT_TEMPLATE)),
    max_seq_len=np.array(int(MAX_SEQ_LEN)),
)
print(f"saved mean_probe.npz  |  layer {best_layer}  |  dim {best_probe.direction.shape[0]}  |  model {repo_id}")
print("-> copy to Aletheias-Quest-Competition/submissions/mean_probe.npz")

## 🧠 Takeaways & next steps

- **The probe is closed-form.** `direction = mean(deceptive) - mean(honest)`, scored by projection — no hyperparameters, no fitting loop. The optional standardization and midpoint `bias` don't change AUROC.
- **Transfer is the score that matters.** A layer that aces in-distribution can collapse — or invert below 0.5 — on a held-out scenario. Always select the layer on validation and report on test.
- **The export is the deliverable.** `mean_probe.npz` is what ships to the competition; the submission notebook reproduces this notebook's extraction exactly and only swaps in the eval transcripts.

Ideas to push on:

- **Rotate the held-out config.** Treat a single held-out scenario as one data point, not a verdict. Rotate which config is held out to get a fuller transfer picture.
- **Swap the model.** Point `repo_id` / `MODEL_NAME` at any NDIF-hosted model. For a text-only model (e.g. Llama-3.3-70B) use `LanguageModel`, read activations at `model.model.layers[L].output[0]` (no `.language_model` nesting), and update the tokenizer's end-of-turn token. Re-export the `.npz` — and update the submission's `MODEL_ID`/extraction to match.
- **Test against the imposters.** The submission traces the base model *with a red-team LoRA adapter* (`peft=`). A probe trained on clean-base activations is off-distribution there — measuring that gap (and training on imposter activations) is a strong direction.
- **Compare detectors.** The mean probe is a baseline; difference-of-means is a special case of broader contrastive and robust-probing methods.

---
*Clear all cell outputs before committing or sharing this notebook.*